In [ ]:
import matplotlib.colors as clr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from mcDETECT.utils import *
from scipy import sparse
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

: 

## Settings

In [3]:
nuc_color = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]
cyto_color = ["#F56867", "#FEB915", "#C798EE", "#59BE86", "#7495D3", "#6D1A9C", "#15821E", "#3A84E6", "#997273", "#D1D1D1"]
color_cts = clr.LinearSegmentedColormap.from_list("magma", ["#000003", "#3B0F6F", "#8C2980", "#F66E5B", "#FD9F6C", "#FBFCBF"], N=256)

# Xenium_5K_BC: (7, 12), (0.6, 1.0)
# Xenium_5K_OC: (8, 6), (0.65, 1.0)
sample = "Xenium_5K_LC"

## Read data

In [4]:
# Paths
intermediate_path = f"../../data/{sample}/intermediate_data/"
processed_path = f"../../data/{sample}/processed_data/"
output_path = f"../../output/1_exploration/{sample}/"

# Read data
adata = sc.read_h5ad(intermediate_path + "adata.h5ad")
cell_ids = np.load(processed_path + "cell_ids.npy", allow_pickle=True)
assert adata.shape[0] > cell_ids.shape[0], "cell_ids should be a subset of adata.obs_names"
adata_tumor = adata[adata.obs["cell_id"].isin(cell_ids)].copy()

adata_tumor_nuc = sc.read_h5ad(output_path + "adata_nuclear_embedded.h5ad")
adata_tumor_cyto = sc.read_h5ad(output_path + "adata_cytoplasmic_embedded.h5ad")

assert adata_tumor.shape[0] == adata_tumor_nuc.shape[0], "adata_tumor and adata_tumor_nuc should have the same number of cells"
assert adata_tumor_nuc.shape[0] == adata_tumor_cyto.shape[0], "Nuclear and cytoplasmic embeddings should have the same number of cells"

## Clustering

In [8]:
# Clustering
res_nuc, res_cyto = 0.5, 0.45
sc.tl.louvain(adata_tumor_nuc, resolution = res_nuc, key_added = "subtype_louvain")
sc.tl.louvain(adata_tumor_cyto, resolution = res_cyto, key_added = "subtype_louvain")

adata_tumor.obs["subtype_louvain_nuc"] = adata_tumor_nuc.obs["subtype_louvain"].values
adata_tumor.obs["subtype_louvain_cyto"] = adata_tumor_cyto.obs["subtype_louvain"].values

print(f"Number of clusters (nuclear): {adata_tumor.obs['subtype_louvain_nuc'].nunique()}")
print(f"Number of clusters (cytoplasmic): {adata_tumor.obs['subtype_louvain_cyto'].nunique()}")

Number of clusters (nuclear): 6
Number of clusters (cytoplasmic): 6


In [9]:
# UMAP plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sc.pl.umap(adata_tumor_nuc, color="subtype_louvain", ax=axes[0], show=False, title="Nuclear expression")
sc.pl.umap(adata_tumor_cyto, color="subtype_louvain", ax=axes[1], show=False, title="Cytoplasmic expression")
plt.savefig(output_path + "umap_clustering.png", dpi=300, bbox_inches="tight")
plt.close()

In [10]:
# ARI
df = pd.DataFrame({"subtype_louvain_nuc": adata_tumor.obs["subtype_louvain_nuc"], "subtype_louvain_cyto": adata_tumor.obs["subtype_louvain_cyto"]})
df.to_csv(output_path + "clustering_results.csv", index=False)

ari = adjusted_rand_score(adata_tumor.obs["subtype_louvain_nuc"], adata_tumor.obs["subtype_louvain_cyto"])
print(f"Adjusted Rand Index (ARI) between nuclear and cytoplasmic clustering: {ari:.4f}")

Adjusted Rand Index (ARI) between nuclear and cytoplasmic clustering: 0.1801


In [11]:
# Scatter plot
for label, palette in zip(["subtype_louvain_nuc", "subtype_louvain_cyto"], [nuc_color, cyto_color]):
    sc.set_figure_params(figsize = (8, 6))
    ax = sc.pl.scatter(adata_tumor, x = "global_x", y = "global_y", color = label, palette = palette, size = 2, alpha = 1, show = False)
    ax.grid(False)
    plt.savefig(output_path + f"scatter_{label}.png", dpi = 300, bbox_inches = "tight")
    plt.close()

## DE analysis

In [12]:
# Within-compartment DE analysis
for adata_tmp, label in zip([adata_tumor_nuc, adata_tumor_cyto], ["nuclear", "cytoplasmic"]):
    
    sc.tl.rank_genes_groups(adata_tmp, "subtype_louvain", method="wilcoxon")
    
    names = adata_tmp.uns["rank_genes_groups"]["names"]
    names = pd.DataFrame(names)
    logfc = adata_tmp.uns["rank_genes_groups"]["logfoldchanges"]
    logfc = pd.DataFrame(logfc)
    pvals = adata_tmp.uns["rank_genes_groups"]["pvals"]
    pvals = pd.DataFrame(pvals)

    for i in adata_tmp.obs["subtype_louvain"].unique():
        df = {"names": names[i], "logfc": logfc[i], "pvals": pvals[i]}
        df = pd.DataFrame(df)
        df = df[df["logfc"] >= 0]
        df = df[df["pvals"] <= 0.05]
        df = df.sort_values(by = ["logfc"], ascending = False)
        file = output_path + f"within_compartment_{label}_cluster_{i}.csv"
        df.to_csv(file, index = False)

In [13]:
# Cross compartments DE analysis
adata_tumor_cyto.obs["subtype_louvain_nuc"] = adata_tumor_nuc.obs["subtype_louvain"].values

sc.tl.rank_genes_groups(adata_tumor_cyto, "subtype_louvain_nuc", method="wilcoxon")
    
names = adata_tumor_cyto.uns["rank_genes_groups"]["names"]
names = pd.DataFrame(names)
logfc = adata_tumor_cyto.uns["rank_genes_groups"]["logfoldchanges"]
logfc = pd.DataFrame(logfc)
pvals = adata_tumor_cyto.uns["rank_genes_groups"]["pvals"]
pvals = pd.DataFrame(pvals)

for i in adata_tumor_cyto.obs["subtype_louvain_nuc"].unique():
    df = {"names": names[i], "logfc": logfc[i], "pvals": pvals[i]}
    df = pd.DataFrame(df)
    df = df[df["logfc"] >= 0]
    df = df[df["pvals"] <= 0.05]
    df = df.sort_values(by = ["logfc"], ascending = False)
    file = output_path + f"cross_compartment_{label}_cluster_{i}.csv"
    df.to_csv(file, index = False)

In [18]:
# Within-compartment dotplot
# nuc_markers = {"cluster_0": ["ANKRD30A", "NOTCH2NLA", "CTTN", "CCND1", "PPFIA1"],
#                "cluster_1": ["XBP1", "CA12"],
#                "cluster_2": ["NR0B1", "HOXB4", "PEG10", "BMPR1B", "EEF1A2"],
#                "cluster_3": [],
#                "cluster_4": ["TYMS", "BIRC5", "CDK1", "RAD54L", "TOP2A"],
#                "cluster_5": []}
# nuc_markers = {"cluster_0": ["EEF1G"],
#                "cluster_1": ["LAPTM4B", "CP", "H19", "MUC16", "SULF1"],
#                "cluster_2": ["SNHG15", "PVT1", "KCNMA1", "KCNC3", "NOTCH2NLA"],
#                "cluster_3": ["BIRC5", "TOP2A", "CDK1", "TPX2", "TYMS"],
#                "cluster_4": ["TFPI2", "VEGFA", "SLC2A1", "NDRG1", "HSPA6"],
#                "cluster_5": []}
nuc_markers = {"cluster_0": ["IGSF9B", "SSX2", "MC1R", "HSF4", "PABPC1L"],
               "cluster_1": ["OLFML3", "TSPAN8", "MALL", "EEF1G", "NDRG1"],
               "cluster_2": ["CSF1R", "ALOX5", "CLEC5A", "CYBB", "SPI1"],
               "cluster_3": ["HHLA2", "FOLR1", "PDIA4", "EPDR1", "LAMP1"],
               "cluster_4": ["SLC6A3", "SORCS2", "ILDR2", "NELL1", "LPCAT1"],
               "cluster_5": ["GRP", "MEG3", "DLK1", "SLC6A5", "ASCL1"]}
nuc_markers_list = [gene for cluster in nuc_markers.values() for gene in cluster]

sc.set_figure_params(scanpy=True, fontsize=10)
fig = sc.pl.dotplot(adata_tumor_nuc, nuc_markers_list, groupby = "subtype_louvain", log = True, cmap = "Reds", standard_scale = "var", swap_axes = True, figsize = (5, 10), show = False)
plt.savefig(output_path + "within_compartment_nuclear_dotplot.jpeg", dpi = 300, bbox_inches = "tight")
plt.close()

# cyto_markers = {"cluster_0": ["TSPAN13", "EEF1G", "CA12", "CCND1", "YWHAZ"],
#                "cluster_1": [],
#                "cluster_2": ["TUBB", "DDR1", "SERPINA3", "PARK7", "TMEM97"],
#                "cluster_3": ["H3F3B"],
#                "cluster_4": ["PEG10"],
#                "cluster_5": ["NR0B1", "EEF1A2", "CYP24A1", "HOXB4", "BMPR1B"]}
# cyto_markers = {"cluster_0": ["FAM107A"],
#                "cluster_1": ["CP", "MUC16", "APP", "CXCL6", "IL1R1"],
#                "cluster_2": ["HNRNPD", "MYBL2", "ADAMTS1", "UCHL1", "SMC4"],
#                "cluster_3": ["BIRC5", "TPX2", "TOP2A", "CENPF", "CDK1"],
#                "cluster_4": ["KPNA2", "CDC20", "CCNB1", "CDCA2", "CENPA"],
#                "cluster_5": ["TFPI2", "HSPA6", "SLC2A1", "NDRG1", "P4HA1"]}
cyto_markers = {"cluster_0": ["HHLA2", "VSTM2L", "NMU", "MX1", "CD44"],
               "cluster_1": ["FYN", "HPGD", "KCNJ15", "PABPC4", "CLIC3"],
               "cluster_2": ["SORCS2", "IRX1", "SLC6A3", "ECEL1", "TSPAN8"],
               "cluster_3": ["SOX2-OT", "CXCR4", "CD68", "MMP14", "ZNF683"],
               "cluster_4": ["TRPM4", "LRRTM1", "CPA6", "HABP2", "MAGEA10"],
               "cluster_5": ["GRP", "SCGN", "DLK1", "SCG2", "ASCL1"]}
cyto_markers_list = [gene for cluster in cyto_markers.values() for gene in cluster]

sc.set_figure_params(scanpy=True, fontsize=10)
fig = sc.pl.dotplot(adata_tumor_cyto, cyto_markers_list, groupby = "subtype_louvain", log = True, cmap = "Reds", standard_scale = "var", swap_axes = True, figsize = (5, 10), show = False)
plt.savefig(output_path + "within_compartment_cytoplasmic_dotplot.jpeg", dpi = 300, bbox_inches = "tight")
plt.close()

In [22]:
# Cross-compartment heatmap
# nuc_markers_cyto = {"cluster_0": ["TSPAN13", "GJA1", "CCND1", "DDR1", "THBS2"],
#                "cluster_1": [],
#                "cluster_2": ["NR0B1", "PEG10", "HOXB4", "EEF1A2", "CYP24A1"],
#                "cluster_3": [],
#                "cluster_4": ["BIRC5", "MYBL2", "CDK1", "TOP2A", "RRM2"],
#                "cluster_5": ["PABPC1L", "LENG8", "PPFIA4", "PLIN4", "MST1"]}
# nuc_markers_cyto = {"cluster_0": [],
#                "cluster_1": ["CXCL6", "H19", "LAPTM4B", "MUC16", "SULF1"],
#                "cluster_2": ["FST", "FAM107A", "FMOD", "RERG", "IL11RA"],
#                "cluster_3": ["TOP2A", "TPX2", "BIRC5", "CENPF", "MYBL2"],
#                "cluster_4": ["HSPA6", "TFPI2", "SLC2A1", "NDRG1", "P4HA1"],
#                "cluster_5": ["U2AF1", "PABPC1L", "LENG8", "PVT1", "GSDMB"]}
nuc_markers_cyto = {"cluster_0": ["HHLA2", "C4A", "CP", "VSTM2L", "C4B"],
               "cluster_1": ["CARD11", "IRX1", "SORCS2", "OLFML3", "TSPAN8"],
               "cluster_2": ["CYBB", "ALOX5", "F13A1", "SPI1", "COMP"],
               "cluster_3": ["NMU", "SEPTIN9", "CD44", "AQP5", "AGR3"],
               "cluster_4": ["LPCAT1", "SLC6A3", "ECEL1", "NELL1", "RASD1"],
               "cluster_5": ["GRP", "DLK1", "SCGN", "ASCL1", "SCG2"]}
nuc_markers_cyto_list = [gene for cluster in nuc_markers_cyto.values() for gene in cluster]

sc.set_figure_params(scanpy=True, fontsize=10)
fig = sc.pl.dotplot(adata_tumor_cyto, nuc_markers_cyto_list, groupby = "subtype_louvain_nuc", log = True, cmap = "Reds", standard_scale = "var", swap_axes = True, figsize = (5, 10), show = False)
plt.savefig(output_path + "cross_compartment_dotplot.jpeg", dpi = 300, bbox_inches = "tight")
plt.close()

In [23]:
len(nuc_markers_cyto_list), len(np.unique(nuc_markers_cyto_list))

(30, 30)

## Stress scores

In [24]:
# ==================== Functions for computing stress scores ==================== #

# z-score normalization
def zscore_series(x):
    mu = np.nanmean(x)
    sd = np.nanstd(x)
    return (x - mu) / (sd + 1e-8)

# expression-based scores
def compute_expression_score(adata, key, target_genes, top_k = 2, binary_subtyping = True):
    
    # filter target genes
    target_genes = [i for i in target_genes if i in adata.var_names]
    
    # extract expression
    X = adata[:, target_genes].X
    if not isinstance(X, np.ndarray):
        X = X.toarray()
        
    # calculate the expression of the k-th top gene
    idx_sorted = np.argsort(-X, axis=1)
    kth_gene_index = idx_sorted[:, top_k - 1].reshape(-1, 1)
    scores = np.take_along_axis(X, kth_gene_index, axis=1).flatten()
    adata.obs[key] = scores
    
    # binary subtyping
    if binary_subtyping:
        thr = adata.obs[key].median()
        adata.obs[f"{key}_subtype"] = pd.Categorical(np.where(adata.obs[key] <= thr, "Low", "High"), categories = ["Low", "High"], ordered = True)
    
    return adata

# mechanical crowding score
def compute_mechanical_crowding(adata_tumor, key, k = 10, binary_subtyping = True):
    
    # coordinates
    XY_tum = np.c_[adata_tumor.obs["global_x"].values, adata_tumor.obs["global_y"].values]

    # kNN on all cells
    nbrs = NearestNeighbors(n_neighbors = k, algorithm = "kd_tree").fit(XY_tum)

    # distances from tumor cells to all cells
    dist, idx = nbrs.kneighbors(XY_tum, return_distance = True)

    # mean distance to k nearest neighbors
    mean_dist = dist.mean(axis = 1)
    mean_dist = np.maximum(mean_dist, 1e-6)

    # inverse distance
    crowding_raw = 1.0 / mean_dist
    adata_tumor.obs[f"{key}_raw"] = crowding_raw

    # z-score normalization
    adata_tumor.obs[key] = np.nan
    adata_tumor.obs.loc[:, key] = zscore_series(adata_tumor.obs.loc[:, f"{key}_raw"].values)
    del adata_tumor.obs[f"{key}_raw"]
    
    # symmetric clipping
    vals = adata_tumor.obs[key].to_numpy()
    vmin, vmax = np.nanmin(vals), np.nanmax(vals)
    assert vmin < 0 and vmax > 0, f"Expected vmin < 0 and vmax > 0, got vmin = {vmin}, vmax = {vmax}"
    clip_val = np.min(np.abs((vmin, vmax)))
    adata_tumor.obs[f"{key}_clipped"] = adata_tumor.obs[key].clip(-clip_val, clip_val)
    
    if binary_subtyping:
        thr = adata_tumor.obs[key].median()
        adata_tumor.obs[f"{key}_subtype"] = pd.Categorical(np.where(adata_tumor.obs[key] <= thr, "Low", "High"), categories = ["Low", "High"], ordered = True)
    
    return clip_val, adata_tumor

# immune attack scores
def compute_spatial_score(adata_all, adata_tumor, key, source_col = "cell_type_merged", source_values = ("T cell",), k = 20, radius = 50.0, sigma = None, binary_subtyping = True, neighbor_weight_col = None):

    # select source cells
    if isinstance(source_values, (list, tuple, set)):
        src_mask = adata_all.obs[source_col].isin(source_values)
    else:
        src_mask = adata_all.obs[source_col] == source_values
    src_idx = np.where(src_mask.values)[0]

    # coordinates
    # XY_src = np.c_[adata_all.obs.loc[adata_all.obs.index[src_idx], "global_x"].values, adata_all.obs.loc[adata_all.obs.index[src_idx], "global_y"].values]
    XY_src = np.c_[adata_all.obs["global_x"].values[src_idx],  adata_all.obs["global_y"].values[src_idx]]
    XY_tum = np.c_[adata_tumor.obs["global_x"].values, adata_tumor.obs["global_y"].values]
    
    # number of neighbors within radius
    tree = cKDTree(XY_src)
    neighbor_counts = tree.query_ball_point(XY_tum, r = radius, return_length = True)
    adata_tumor.obs[f"{key}_neighbor_counts"] = neighbor_counts
    adata_tumor.obs[f"log_{key}_neighbor_counts"] = np.log1p(neighbor_counts)

    if neighbor_weight_col is not None:
        
        # retrieve weights for source cells
        expr = adata_all[src_idx, neighbor_weight_col].X
        if not isinstance(expr, np.ndarray):
            expr = expr.toarray()
        W = expr.mean(axis = 1).ravel()
        W = np.where(np.isfinite(W), W, 0.0)

        # kNN search for distances and indices
        n_neighbors = min(k, XY_src.shape[0])
        nbrs = NearestNeighbors(n_neighbors = n_neighbors, algorithm = "kd_tree").fit(XY_src)
        dist, nn_ind = nbrs.kneighbors(XY_tum, return_distance = True)

        # apply radius mask
        within = dist <= radius

        # Gaussian kernel
        if sigma is None:
            sigma = radius / 2.0
        K = np.exp(-(dist ** 2) / (2.0 * (sigma ** 2)))
        K = K * within

        # apply weights
        W_neighbors = W[nn_ind]  # shape: (n_tumor, n_neighbors)
        contrib = K * W_neighbors

        # aggregate: mean over neighbors
        alpha = 0.5
        denom = np.maximum(within.sum(axis = 1, keepdims = True), 1) ** alpha
        raw_score = (contrib.sum(axis = 1, keepdims = True) / denom).ravel()
        adata_tumor.obs[f"{key}_weighted_raw"] = raw_score

        # z-score normalization
        adata_tumor.obs[f"{key}_weighted"] = np.nan
        adata_tumor.obs.loc[:, f"{key}_weighted"] = zscore_series(adata_tumor.obs.loc[:, f"{key}_weighted_raw"].values)
        del adata_tumor.obs[f"{key}_weighted_raw"]
    
    # binary subtyping
    if binary_subtyping:
        counts = np.array(neighbor_counts)
        adata_tumor.obs[f"{key}_subtype"] = pd.Categorical(np.where(counts == 0, "Away", "Close"), categories = ["Away", "Close"], ordered = True)
        if neighbor_weight_col is not None:
            thr = adata_tumor.obs[f"{key}_weighted"].median()
            adata_tumor.obs[f"{key}_weighted_subtype"] = pd.Categorical(np.where(adata_tumor.obs[f"{key}_weighted"] <= thr, "Low", "High"), categories = ["Low", "High"], ordered = True)

    return adata_tumor

### 2.1 Hypoxia and heat shock

In [25]:
gene_programs = {"hypoxia": ["HIF1A", "EPAS1", "NFE2L2", "CREB1", "RELA", "RELB", "NFKB1", "NFKB2"],
                 "heat_shock": ["HSP90AA1", "HSP90AB1", "HSPA1A", "HSPA1B", "HSPA6", "HSPA8", "HSPH1", "DNAJB1", "HSPB1", "HSPD1", "HSPE1"]}

for key, geneset in gene_programs.items():
    geneset = [gene for gene in geneset if gene in adata_tumor.var_names]
    for i in range(len(geneset)):
        adata_tumor_nuc = compute_expression_score(adata_tumor_nuc, key = f"{key}_{i + 1}_genes", target_genes = geneset, top_k = i + 1)
        adata_tumor_cyto = compute_expression_score(adata_tumor_cyto, key = f"{key}_{i + 1}_genes", target_genes = geneset, top_k = i + 1)

assert "hypoxia_1_genes" in adata_tumor_nuc.obs.columns, "Expected 'hypoxia_1_genes' in adata_tumor_nuc.obs.columns"
assert "hypoxia_1_genes" in adata_tumor_cyto.obs.columns, "Expected 'hypoxia_1_genes' in adata_tumor_cyto.obs.columns"
assert "heat_shock_1_genes" in adata_tumor_nuc.obs.columns, "Expected 'heat_shock_1_genes' in adata_tumor_nuc.obs.columns"
assert "heat_shock_1_genes" in adata_tumor_cyto.obs.columns, "Expected 'heat_shock_1_genes' in adata_tumor_cyto.obs.columns"

### 2.2 Mechanical score

In [26]:
key = "mechanical"

clip_val_nuc, adata_tumor_nuc = compute_mechanical_crowding(adata_tumor_nuc, key = key, k = 10, binary_subtyping = True)
clip_val_cyto, adata_tumor_cyto = compute_mechanical_crowding(adata_tumor_cyto, key = key, k = 10, binary_subtyping = True)

assert "mechanical_clipped" in adata_tumor_nuc.obs.columns, "Expected 'mechanical_clipped' in adata_tumor_nuc.obs.columns"
assert "mechanical_clipped" in adata_tumor_cyto.obs.columns, "Expected 'mechanical_clipped' in adata_tumor_cyto.obs.columns"

### 2.3 Immune attack

In [27]:
# Define proximity stress programs
spatial_programs = {"immune_cell_proximity": ["CD4+ T cell", "CD8+ T cell", "T cell", "B cell", "Dendritic cell", "Myeloid cell", "Mast cell"],
                    "tcell_proximity": ["CD4+ T cell", "CD8+ T cell", "T cell"],
                    "tcell_attack": ["CD4+ T cell", "CD8+ T cell", "T cell"]}

for key, src_vals in spatial_programs.items():
    if key == "tcell_attack":
        adata_tumor_nuc = compute_spatial_score(adata, adata_tumor_nuc, key = key, source_col = "cell_type_merged", source_values = src_vals, k = 20, radius = 100, binary_subtyping = True, neighbor_weight_col = ["GZMB", "GZMK", "GZMA"])
        adata_tumor_cyto = compute_spatial_score(adata, adata_tumor_cyto, key = key, source_col = "cell_type_merged", source_values = src_vals, k = 20, radius = 100, binary_subtyping = True, neighbor_weight_col = ["GZMB", "GZMK", "GZMA"])
    else:
        adata_tumor_nuc = compute_spatial_score(adata, adata_tumor_nuc, key = key, source_col = "cell_type_merged", source_values = src_vals, k = 20, radius = 50, binary_subtyping = True)
        adata_tumor_cyto = compute_spatial_score(adata, adata_tumor_cyto, key = key, source_col = "cell_type_merged", source_values = src_vals, k = 20, radius = 50, binary_subtyping = True)

assert "immune_cell_proximity_subtype" in adata_tumor_nuc.obs.columns, "Expected 'immune_cell_proximity_subtype' in adata_tumor_nuc.obs.columns"
assert "immune_cell_proximity_subtype" in adata_tumor_cyto.obs.columns, "Expected 'immune_cell_proximity_subtype' in adata_tumor_cyto.obs.columns"
assert "tcell_proximity_subtype" in adata_tumor_nuc.obs.columns, "Expected 'tcell_proximity_subtype' in adata_tumor_nuc.obs.columns"
assert "tcell_proximity_subtype" in adata_tumor_cyto.obs.columns, "Expected 'tcell_proximity_subtype' in adata_tumor_cyto.obs.columns"
assert "tcell_attack_subtype" in adata_tumor_nuc.obs.columns, "Expected 'tcell_attack_subtype' in adata_tumor_nuc.obs.columns"
assert "tcell_attack_subtype" in adata_tumor_cyto.obs.columns, "Expected 'tcell_attack_subtype' in adata_tumor_cyto.obs.columns"

In [28]:
rename_mapping = {
    "hypoxia_2_genes": "hypoxia_score",
    "hypoxia_2_genes_subtype": "hypoxia_subtype",
    "heat_shock_2_genes": "heat_shock_score",
    "heat_shock_2_genes_subtype": "heat_shock_subtype",
}

keep_cols = {"hypoxia_score", "hypoxia_subtype", "heat_shock_score", "heat_shock_subtype",}

for adata in [adata_tumor_nuc, adata_tumor_cyto]:
    adata.obs = adata.obs.rename(columns=rename_mapping)
    cols_to_drop = [c for c in adata.obs.columns if ((c.startswith("hypoxia_") or c.startswith("heat_shock_")) and c not in keep_cols)]
    adata.obs = adata.obs.drop(columns=cols_to_drop)

### Correlations

In [29]:
df_nuc = pd.DataFrame({
    "cell_id": adata_tumor_nuc.obs["cell_id"].copy(),
    "subtype_louvain": adata_tumor_nuc.obs["subtype_louvain"].copy(),
    "hypoxia_score": adata_tumor_nuc.obs["hypoxia_score"].copy(),
    "hypoxia_subtype": adata_tumor_nuc.obs["hypoxia_subtype"].copy(),
    "heat_shock_score": adata_tumor_nuc.obs["heat_shock_score"].copy(),
    "heat_shock_subtype": adata_tumor_nuc.obs["heat_shock_subtype"].copy(),
    "mechanical_score": adata_tumor_nuc.obs["mechanical_clipped"].copy(),
    "mechanical_subtype": adata_tumor_nuc.obs["mechanical_subtype"].copy(),
    "immune_cell_proximity_score": adata_tumor_nuc.obs["log_immune_cell_proximity_neighbor_counts"].copy(),
    "immune_cell_proximity_subtype": adata_tumor_nuc.obs["immune_cell_proximity_subtype"].copy(),
    "tcell_poximity_score": adata_tumor_nuc.obs["log_tcell_proximity_neighbor_counts"].copy(),
    "tcell_proximity_subtype": adata_tumor_nuc.obs["tcell_proximity_subtype"].copy(),
    "tcell_attack_score": adata_tumor_nuc.obs["tcell_attack_weighted"].copy(),
    "tcell_attack_subtype": adata_tumor_nuc.obs["tcell_attack_subtype"].copy(),
})

df_cyto = pd.DataFrame({
    "cell_id": adata_tumor_cyto.obs["cell_id"].copy(),
    "subtype_louvain": adata_tumor_cyto.obs["subtype_louvain"].copy(),
    "hypoxia_score": adata_tumor_cyto.obs["hypoxia_score"].copy(),
    "hypoxia_subtype": adata_tumor_cyto.obs["hypoxia_subtype"].copy(),
    "heat_shock_score": adata_tumor_cyto.obs["heat_shock_score"].copy(),
    "heat_shock_subtype": adata_tumor_cyto.obs["heat_shock_subtype"].copy(),
    "mechanical_score": adata_tumor_cyto.obs["mechanical_clipped"].copy(),
    "mechanical_subtype": adata_tumor_cyto.obs["mechanical_subtype"].copy(),
    "immune_cell_proximity_score": adata_tumor_cyto.obs["log_immune_cell_proximity_neighbor_counts"].copy(),
    "immune_cell_proximity_subtype": adata_tumor_cyto.obs["immune_cell_proximity_subtype"].copy(),
    "tcell_poximity_score": adata_tumor_cyto.obs["log_tcell_proximity_neighbor_counts"].copy(),
    "tcell_proximity_subtype": adata_tumor_cyto.obs["tcell_proximity_subtype"].copy(),
    "tcell_attack_score": adata_tumor_cyto.obs["tcell_attack_weighted"].copy(),
    "tcell_attack_subtype": adata_tumor_cyto.obs["tcell_attack_subtype"].copy(),
})

In [30]:
# Quantitative scores
scores = ["hypoxia_score", "heat_shock_score", "mechanical_score", "immune_cell_proximity_score", "tcell_poximity_score", "tcell_attack_score"]

for df, label in zip([df_nuc, df_cyto], ["nuclear", "cytoplasmic"]):
    subtype_order = sorted(df["subtype_louvain"].dropna().unique(), key=lambda x: int(x))
    for score in scores:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df, x="subtype_louvain", y=score, order=subtype_order, showfliers=False, width=0.4)
        plt.xlabel("Subtype")
        plt.ylabel("Score")
        plt.title(f"{score.replace('_', ' ').title()}")
        plt.savefig(output_path + f"{label}_{score}_boxplot.png", dpi=300, bbox_inches="tight")
        plt.close()

In [31]:
# Categorical scores
categorical_scores = ["hypoxia_subtype", "heat_shock_subtype", "mechanical_subtype", "immune_cell_proximity_subtype", "tcell_proximity_subtype", "tcell_attack_subtype"]

for df, label in zip([df_nuc, df_cyto], ["nuclear", "cytoplasmic"]):
    # subtype_order = sorted(df["subtype_louvain"].dropna().unique(), key=lambda x: int(x))
    for score in categorical_scores:
        prop = pd.crosstab(df["subtype_louvain"], df[score], normalize="index")
        prop = prop.loc[sorted(prop.index, key=lambda x: int(x))]
        ax = prop.plot(kind="bar", stacked=True, figsize=(8, 5), width=0.7)
        ax.set_xlabel("Subtype")
        ax.set_ylabel("Proportion")
        ax.set_title(f"{score.replace('_', ' ').title()}")
        ax.set_ylim(0, 1.05)
        ax.legend(title=f"{score.replace('_', ' ').title()}", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.grid(False)
        plt.xticks(rotation=0)
        plt.savefig(output_path + f"{label}_{score}_stacked_barplot.png", dpi=300, bbox_inches="tight")
        plt.close()